# zi2zi-JiT · 字体训练 CloudStudio Notebook（一键版）

基于 **zi2zi-JiT**（ICW fork）。参照 HanziGen 的 `hanzigen_cloudstudio.ipynb` 设计：

- 训练所需参数全部集中在 **Cell 0**，其余代码 Cell 不用改
- 每个代码 Cell 前都有 **markdown 说明**（作用 / 输入 / 输出 / 产物位置）
- 导入（预训练模型、目标字体）与导出（PNG zip 下载）遵循腾讯云**可视化目录**约定

**使用流程：**

1. 把预训练模型 `zi2zi-JiT-B-16.pth`（README 下载链接）上传到 `models/`
2. 把要学的新字体（.ttf/.otf）上传到 `fonts/`
3. 修改 **Cell 0**（其余 Cell 从头到尾依次运行即可）
4. 跑完后在 `exports/` 或 Cell 6 的下载按钮拿到 zip

**目录约定（腾讯云"可视化 / 不可视化目录"）：**

| 目录 | 类型 | 内容 |
|---|---|---|
| `fonts/` | ✅ 可视化 | 目标字体（上传）、参照字体（上传） |
| `models/` | ✅ 可视化 | 预训练模型（上传） |
| `data/` | ✅ 可视化 | 自动生成的数据集 |
| `outputs/` | ✅ 可视化 | 训练产物 + 推理 PNG + 补集 PNG |
| `exports/` | ✅ 可视化 | 最终打包 zip（可直接下载） |
| `/root`、`/tmp`、conda 缓存等 | ❌ 不可视化 | 文件树看不到，实例回收会丢，**不要放重要数据** |

> 建议先在 Cloud Studio 里按 `environment.yaml` 建好 Python 3.10 + PyTorch 环境；
> Cell 1 也会自动检查并补装依赖（首次运行较慢）。

---

## 🔧 机时节省指南：哪些 Cell 用 CPU / GPU

云平台 GPU 机时贵，下表标注各 Cell 的**实际计算负载**，便于分机型/分时段跑。

| Cell | 名称 | CPU 能跑？ | 必须 GPU？ | 耗时量级 | 说明 |
|---|---|---|---|---|---|
| **0** | 参数预设 | ✅ | — | 秒级 | 纯变量赋值 |
| **1** | 环境初始化 + 自检 + 调优 | ✅ | ⚠️ 部分 | 秒~分钟 | `AUTO_TUNE` 探测显存需 GPU；无 GPU 时走 fallback 估算（不准但不报错）。依赖首次 `pip install` 约 5~10 分钟 |
| **2** | 数据准备（渲染字形） | ✅ | ❌ | 分钟~几十分钟 | fontTools + cv2 渲染，**纯 CPU**；字数越多越久（gbk 全量 2 万字约 10~30 分钟） |
| **3** | LoRA 训练 | ❌ | ✅✅ | **小时级** | EDM 扩散 + LoRA 反向传播 + 在线 FID，**全程 GPU**。最耗机时的一步 |
| **4** | 推理生成（测试集字） | ❌ | ✅ | 分钟级 | 扩散采样，**必须 GPU** |
| **5** | 缺失字补集 | ❌ | ✅ | 分钟级 | 扩散采样，**必须 GPU**；缺失字越多越久 |
| **6** | 导出打包 + 下载 | ✅ | ❌ | 秒~分钟 | zip + base64，**纯 CPU** |

**建议的省机时工作流：**

1. **阶段一（CPU 低配机即可）**：跑 Cell 0 → 1 → 2，把数据集 `data/<字体>/` 生成好
2. **阶段二（GPU 高配机）**：换/开 GPU 实例，跑 Cell 0 → 1（跳过数据准备 `DO_DATA_PREP=False`）→ 3 → 4 → 5
3. **阶段三（CPU 低配机即可）**：跑 Cell 6 打包下载

> 三个阶段之间实例可关停/切换机型，只要 `data/` 和 `outputs/` 在可视化目录里不丢。
> Cell 3 是唯一真正烧 GPU 机时的步骤，其余尽量在 CPU 机器上做。

## Cell 0 · 参数预设（训练新字体只改这一个 Cell）

**🖥️ 推荐配置：任意，纯 CPU 最低配（1核2G）即可** —— 本 Cell 只做变量赋值，无计算负载，可在任何机器上跑。

> 每个参数都标了 **✅建议改 / ⛔不建议改 / 🤖自动计算**。
> 带 `*` 的"须与预训练模型一致"，改了会导致加载预训练模型失败。

| 参数 | 作用 | 建议 |
|---|---|---|
| **运行开关** | | |
| `DO_DATA_PREP` | 是否重新生成数据集 | ✅ 首次/换字体=True；已有数据集可设 False 跳过 |
| `DO_TRAIN` | 是否训练 | ✅ 同上 |
| `DO_GENERATE` | 是否推理生成 PNG | ✅ 同上 |
| `DO_MISSING_GEN` | 是否生成"缺失字补集" | ✅ 同上（补集=按 CHARSET 补齐目标字体缺的字） |
| `DO_EXPORT` | 是否打包导出 | ✅ 同上 |
| **导入（素材放可视化目录）** | | |
| `TARGET_FONTS` | **要学的新字体**（放 `fonts/`） | 🔴 **必改**：改成你的字体文件名，如 `["fonts/zhufengti.TTF"]`（可多个=多字体） |
| `SOURCE_FONT` | 参照字体，提供"源字形" | 留空自动挑；推荐显式填 `fonts/jigmo/jigmo.ttf` |
| `BASE_CHECKPOINT` | 预训练模型 | 🔴 **必改**：与 `MODEL`/`CFG` 配套——B 模型 → `models/zi2zi-JiT-B-16.pth` + `JiT-B/16` + `CFG=2.6`；L 模型 → 对应 `JiT-L/16` + `CFG=2.4` |
| `FONTS_DIR` … `EXPORTS_DIR` | 各目录 | ⛔ 不建议改 |
| **数据准备** | | |
| `CHARSET` | 字符集：简中`gb2312`（6763 字）/ 繁中`big5` / 日文`jisx0208` / 韩文`ksx1001` / **真 GBK：`gbk`（20,902 字，同 HanziGen，已内置）** | ✅ 按你的字体面向地区改。用真 GBK 补字请同步 `NUM_CHARS=30000`、`MAX_CHARS_PER_FONT=None`（训练会更久） |
| `TRAIN_CHARS_PER_FONT` | 每个字体从 CHARSET 抽多少个字符进**训练集** | ✅ 字符越多风格学得越全；补集效果最佳 → **≥ 字符集大小**（gb2312 用 6763）；此时 test 集自动降级为训练字展示 |
| `TEST_CHARS_PER_FONT` | 每个字体抽多少个**训练未见过**的字符进**测试集**（只评估泛化，不参与训练） | ⛔ 8 够用；训练覆盖全部交集字时会自动降级为展示用（非泛化指标） |
| `RESOLUTION` | 渲染分辨率 | ⛔ 须与 `IMG_SIZE` 一致（256） |
| `TRAIN_SEED` / `TEST_SEED` | 随机种子 | ⛔ 复现用 |
| `NUM_WORKERS_DATA_PREP` | 数据生成并行数 | ⛔ 不建议改 |
| **LoRA 训练** | | |
| `MODEL`* | JiT-B/16 或 JiT-L/16 | ⛔ 须与预训练模型一致 |
| `IMG_SIZE`* | 训练分辨率 | ⛔ 须与预训练模型一致（256） |
| `NUM_FONTS`* | 字体嵌入维度 | ⛔ 必须 = 预训练模型（1000），改了就加载失败 |
| `NUM_CHARS` | 字符嵌入上界（只需 ≥ 数据集字符数） | ✅ 默认 20000 够用；`CHARSET="gbk"` 时**必须** 30000 |
| `MAX_CHARS_PER_FONT` | 训练时每字体实际使用字符数的上限（None=全部） | ✅ 补集建议 `None`（让 TRAIN_CHARS_PER_FONT 抽的字全部参与训练） |
| `LORA_R` / `LORA_ALPHA` | LoRA 容量（决定新字体学得多细） | ✅ **风格化明显→64；规整→默认32**；显存几乎不受影响 |
| `LORA_TARGETS` / `LORA_DROPOUT` / `PROJ_DROPOUT` | LoRA 注入层 / 丢弃率 | ⛔ 不建议改 |
| `EPOCHS` | 训练轮数 | ✅ **风格化明显→300+；规整→200 足够** |
| `BLR` / `MIN_LR` / `WARMUP_EPOCHS` | 学习率 | ⛔ README 推荐值 |
| `SAVE_LAST_FREQ` | 保存 checkpoint 频率 | ⛔ 不建议改 |
| `P_MEAN` / `P_STD` / `NOISE_SCALE` | 扩散噪声分布（**训练期参数，固化进 checkpoint，生成阶段不能覆盖**） | ⛔ 一般不调。EDM 经验：笔画乱/错字多→`NOISE_SCALE` 降到 0.8；模糊/缺细节→提到 1.2；`P_MEAN`/`P_STD` 保持 -0.8/0.8 |
| `CFG` | 引导强度（越大风格越浓，过大会糊） | ✅ **风格化明显→3.5~4.0；默认 2.6**（JiT-L/16 用 2.4） |
| `ONLINE_EVAL` / `EVAL_STEP_FOLDERS` / `EVAL_FREQ` / `NUM_IMAGES` | 训练中在线评估 | ⛔ 不建议改 |
| `BATCH_SIZE` / `GEN_BSZ` | 训练/推理批量 | 🤖 **自动计算**（AUTO_TUNE=True 时被覆盖，不用手动改） |
| **硬件自动调优** | | |
| `AUTO_TUNE` | 自动推算 batch_size 等 | ✅ 默认 True 即可 |
| `TUNE_METHOD` | `probe`=运行时实测（最准） / `table`=标定表估算 | ✅ 推荐 `probe` |
| `TUNE_RESERVE` / `TUNE_MAX_*` | 调优安全系数/上限 | ⛔ 不建议改 |
| **推理生成** | | |
| `GENERATE_*` | 生成张数/批量/CFG/采样方法 | ✅ **效果不好时调**：`GENERATE_CFG` 加大、`GENERATE_SAMPLING_METHOD="heun"`、加大步数 |
| **缺失字补集** | | |
| `DO_MISSING_GEN` / `MISSING_*` | 补集开关与参数 | ✅ 需要补缺字时开；`MISSING_CHARSET` 默认沿用 `CHARSET`。提速：`MISSING_PAIRWISE=""`（省一半写盘）、`MISSING_BATCH_SIZE` 调大（V100 用 128）、或先 `MISSING_NUM_IMAGES=100` 试效果 |
| `MISSING_PAIRWISE` | 对比图开关：`"src_gen"`=源字形\|生成结果（默认，写盘翻倍）/ `""`=只出生成图（省一半 I/O）/ `target_gen` 不可用 | ⚠️ 对比图里的"源字形"来自**参照字体**（不是目标字体有的字——缺失字目标字体根本没有）；只想最快看结果就设 `""` |
| **导出** | | |
| `EXPORT_PREFIX` / `EXPORT_INCLUDE_CHECKPOINT` | zip 名前缀 / 是否含 checkpoint | ⛔ 不建议改 |

**常见场景怎么调：**

| 场景 | 建议 |
|---|---|
| **风格化明显的字体**（行书/草书/手写体/装饰体） | `LORA_R=64`、`EPOCHS=300+`、`CFG=3.5~4.0`、`TRAIN_CHARS_PER_FONT` 尽量覆盖整个 CHARSET |
| **字形规整的字体**（黑体/宋体/楷体） | 默认参数即可，`CFG` 保持 2.6 |
| **生成效果不理想** | 优先试：加大 `GENERATE_CFG` → 换 `heun` 采样 → 加步数 → 加 `EPOCHS`/`LORA_R` |
| **显存不够 / 想跑得快** | 保持 `AUTO_TUNE=True`（会自动收紧 batch）；不要手动把 batch 调大 |
| **只想要补集（缺字补全）** | `DO_TRAIN/DO_GENERATE` 可留 True（需要先训练），补集产物在 `outputs/<字体>/missing_chars/` |

**关于《腾讯云可用配置及建议260819.txt》：** 自动调优负责"**在这台机器上 batch_size 用多少**"；该文档负责"**选哪台机器**"（T4 / V100 / A10 的显存、价格、三阶段选型：阶段一数据准备低配、阶段二训练、阶段三导出）。所以**具体 batch 数值不用再翻文档**（已自动计算），但**选机型时仍可参考文档**。

In [ ]:
# ============================================================
# Cell 0 · 参数预设（训练新字体只需修改本 Cell）
# 每个变量都加了简注；详细说明见上方 markdown 表
# ============================================================
import os

# ---------- 运行开关（本 Cell 跑哪些步骤） ----------
DO_DATA_PREP   = True     # 重新生成数据集（首次/换字体=True；已有数据集可设 False 跳过）
DO_TRAIN       = True     # 训练 LoRA 模型（生成缺失字前必须先训练）
DO_GENERATE    = True     # 生成"指定字符"推理 PNG
DO_MISSING_GEN = True     # 生成"缺失字补集"（按 CHARSET 补齐目标字体缺的字）
DO_EXPORT      = True     # 打包导出 zip（可直接下载）

# ---------- 目录（素材放可视化目录 fonts/、models/，本 Cell 会自动创建） ----------
FONTS_DIR       = os.path.join(os.path.abspath(""), "fonts")    # 字体目录：上传目标字体 + 参照字体
MODELS_DIR      = os.path.join(os.path.abspath(""), "models")   # 模型目录：上传预训练模型
DATA_DIR        = os.path.join(os.path.abspath(""), "data")     # 数据集目录（自动生成）
OUTPUTS_DIR     = os.path.join(os.path.abspath(""), "outputs")  # 训练/推理产物目录（自动生成）
EXPORTS_DIR     = os.path.join(os.path.abspath(""), "exports")  # 最终 zip 导出目录（自动生成）

# ---------- 导入（素材） ----------
SOURCE_FONT     = ""                                 # 参照字体：提供"源字形"内容，必须要有
                                                     # （推荐填 fonts/jigmo/jigmo.ttf；
                                                     #  留空=自动发现 fonts/jigmo/ 或 fonts/ 顶层非目标字体）
TARGET_FONTS    = ["fonts/MyTargetFont.ttf"]         # 要学的新字体（可多个=多字体）<-- 必改
BASE_CHECKPOINT = "models/zi2zi-JiT-B-16.pth"        # 预训练模型（B/L 二选一，文件名须与 MODEL 配套）

# ---------- 数据准备 ----------
CHARSET               = "gb2312"   # 字符集：gb2312(6763)/gbk(20902)/big5/jisx0208/ksx1001
TRAIN_CHARS_PER_FONT  = 500        # Cell2 数据准备：从「源∩目标∩CHARSET」抽多少字生成训练集
                                   # 补集效果最佳 → 尽量大（>= 字符集大小，取全部交集字）
                                   # 只想快速试风格 → 可设小值（但补集里未见过字的质量会下降）
                                   # ⚠️ 需搭配 MAX_CHARS_PER_FONT=None，否则训练时会被截断（见下方 LoRA 训练区）
TEST_CHARS_PER_FONT   = 8          # 每字体抽多少个"训练未见字"进测试集（交集不足时自动降级为训练字展示）
RESOLUTION            = 256        # 渲染分辨率（必须 = IMG_SIZE）
TRAIN_SEED            = 42         # 训练集抽样种子（复现用）
TEST_SEED             = 99999      # 测试集抽样种子（复现用）
NUM_WORKERS_DATA_PREP = 4          # 数据生成并行进程数
CLEAR_OTHER_FONTS_DATA = True      # 训练新字体时是否删除其他字体的 data/<字体>/ 数据集
                                   # True=删除（默认，省磁盘）；False=保留（换回旧字体可免重新生成，但占空间）

# ---------- LoRA 训练 ----------
MODEL              = "JiT-B/16"   # 模型架构（必须与预训练模型配套：JiT-B/16 或 JiT-L/16）
IMG_SIZE           = 256          # 训练分辨率（必须与预训练模型一致）
NUM_FONTS          = 1000         # 字体嵌入维度（必须 = 预训练模型，改了会加载失败）
NUM_CHARS          = 20000        # 字符嵌入上界（>= 数据集字符数即可；真 GBK 请用 30000）
MAX_CHARS_PER_FONT = None          # Cell3 训练：每字体实际参与训练的字数上限
                                   # None=不限制（推荐，用满 TRAIN_CHARS_PER_FONT 生成的全部字）
                                   # ⚠️ 若设数字，必须 >= TRAIN_CHARS_PER_FONT，否则训练时会随机截断：
                                   #    多生成的图白费，且实际训练字数远小于预期（错字多/风格学不会的常見原因）
                                   #    想少训字请直接调小 TRAIN_CHARS_PER_FONT（在 Cell2 阶段就少生成，更省时间）
LORA_R             = 32           # LoRA 秩：容量（风格化明显 -> 64）
LORA_ALPHA         = 32           # LoRA alpha：缩放（一般 = LORA_R）
LORA_TARGETS       = "qkv,proj,w12,w3"   # LoRA 作用的层（qkv/proj=注意力，w12/w3=FFN）
LORA_DROPOUT       = 0.0          # LoRA dropout（防过拟合）
PROJ_DROPOUT       = 0.1          # 投影层 dropout
EPOCHS             = 200          # 训练轮数（风格化明显 -> 300+）
BLR                = 8e-4         # 基础学习率
MIN_LR             = 1e-6         # 最低学习率（训练后期）
WARMUP_EPOCHS      = 1            # 学习率预热轮数
SAVE_LAST_FREQ     = 10           # 每 N 轮保存一次 checkpoint-last.pth

# ---------- checkpoint 历史版本备份开关（默认 None=不备份，只有 checkpoint-last.pth） ----------
# 按"第几次保存 checkpoint"计数，与 SAVE_LAST_FREQ 解耦：
# 调整 SAVE_LAST_FREQ 时备份节奏会自动跟随，两者不会互相冲突。
CKPT_BACKUP_EVERY  = None         # 每第几次保存时额外存一份带轮次号的 checkpoint-epXXXX.pth
                                  # None=不备份（默认）；1=每次保存都备份；2=每 2 次保存备份一次
CKPT_BACKUP_KEEP   = 3            # 历史版本最多保留几个（超出则滚动删除最旧的，防占满磁盘）
SEED               = 42           # 训练随机种子（复现用）
P_MEAN             = -0.8         # 噪声调度均值（EDM 参数，一般不调）
P_STD              = 0.8          # 噪声调度标准差（EDM 参数，一般不调）
NOISE_SCALE        = 1.0          # 噪声缩放（笔画乱/错字多 -> 0.8；太模糊 -> 1.2）
CFG                = 2.6          # 推理引导强度（风格化明显 -> 3.5~4.0）
ONLINE_EVAL        = True         # 训练过程中在线评估
EVAL_STEP_FOLDERS  = True         # 评估图按 step 分文件夹保存
EVAL_FREQ          = 10           # 每 N 轮评估一次
NUM_IMAGES         = 6            # 每字体评估生成几张图
BATCH_SIZE         = 16           # 训练 batch（AUTO_TUNE=True 时会被自动覆盖）
GEN_BSZ            = 16           # 评估 batch（AUTO_TUNE=True 时会被自动覆盖）

# ---------- 硬件自动调优（推算 batch/workers） ----------
AUTO_TUNE           = True        # 自动测显存推算 BATCH_SIZE/GEN_BSZ/NUM_WORKERS
TUNE_METHOD         = "probe"     # probe=实测 / table=标定表
TUNE_RESERVE        = 0.85        # 显存预留比例（0.85=留 15% 余量）
TUNE_MAX_BATCH      = 128         # 推算出的 batch 上限
TUNE_MAX_GEN_BSZ    = 32          # 推算出的评估 batch 上限
TUNE_NUM_WORKERS_CAP = 12         # 数据加载进程数上限

# ---------- 推理生成（None=沿用训练时配置） ----------
GENERATE_NUM_IMAGES         = None    # 每字生成几张图（None=1）
GENERATE_BATCH_SIZE         = 64      # 推理 batch 大小
GENERATE_CFG                = None    # 推理引导强度（None=沿用 CFG）
GENERATE_SAMPLING_METHOD    = None    # 采样方法（None=沿用训练默认）
GENERATE_NUM_SAMPLING_STEPS = None    # 采样步数（None=沿用训练默认）
GENERATE_PAIRWISE           = None    # 对比图开关：None=不出图；"src_gen"=源字形|生成结果；"target_gen"=目标字形|生成结果（传值须带引号，写入 compare/）

# ---------- 缺失字补集 ----------
MISSING_CHARSET         = None           # 补集用哪个字符集（None=沿用 CHARSET）
MISSING_BATCH_SIZE      = 128            # 补集 batch 大小（V100 可调大，减少 batch 切换开销）
MISSING_CFG             = None           # 补集引导强度（None=沿用 CFG）
MISSING_SAMPLING_METHOD = None           # 补集采样方法（None=沿用训练默认）
MISSING_NUM_SAMPLING_STEPS = None        # 补集采样步数（None=沿用训练默认）
MISSING_NUM_IMAGES      = None           # 补几张图（None=全部缺失字）
MISSING_PAIRWISE        = "src_gen"      # 输出"源字形|生成结果"对比图（""=只出生成图，省一半写盘、显著提速）
MISSING_REF_CHARS       = ""             # 逗号分隔的样式参考字（留空自动挑）

# ---------- 导出 ----------
EXPORT_PREFIX             = "zi2zi_jit"  # 导出 zip 文件名前缀
EXPORT_INCLUDE_CHECKPOINT = True         # 是否把训练权重一并打包进 zip

# ---------- 派生路径（自动计算，不用改） ----------
FONT_TAG       = os.path.splitext(os.path.basename(TARGET_FONTS[0]))[0]  # 由目标字体文件名生成的标签
# 数据集按「字体 + 字符集」分目录：切换 CHARSET（gbk<->gb2312）不会互相覆盖，旧的还能留存复用
DATASET_DIR    = os.path.join(DATA_DIR, FONT_TAG, str(CHARSET).lower())  # 本字体+字符集数据集目录
TRAIN_DIR      = os.path.join(DATASET_DIR, "train")  # 训练图像目录
TEST_NPZ_PATH  = os.path.join(DATASET_DIR, "test.npz")  # 测试集（推理用）
OUTPUT_DIR     = os.path.join(OUTPUTS_DIR, FONT_TAG)   # 训练产物目录
GEN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "generated_chars")    # 推理 PNG 目录
MISSING_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "missing_chars")  # 补集 PNG 目录


## Cell 1 · 环境初始化 + 导入检查 + 硬件自动调优

**🖥️ 推荐配置：CPU 进阶级 16核32G（省钱备选 8核16G）**

- 本 Cell 要 import torch、构建探测模型、跑自检，纯 CPU 即可完成；首次 `pip install` 也吃网络和磁盘。
- `AUTO_TUNE=True` 且 `TUNE_METHOD=probe` 时，探测显存需 **GPU**；无 GPU 会自动走 fallback 估算（不准但不报错）。
- **若要用 V100 训练，本 Cell 可直接在 V100 上跑**（自动探测出真实 batch），不必先切回 CPU。

**本 Cell 做什么（无需修改）：**

1. 直接把项目根目录设为 ipynb 所在目录（**不要在腾讯云里 git clone**，clone 出来的目录会不可视化）
2. 检查依赖（缺 torch 时自动 `pip install`，首次运行较慢）
3. 导入检查：目标字体是否存在、自动挑选源字体、预训练模型是否存在（缺什么会报错提示）
4. **配置自检**：`MODEL`↔预训练模型尺寸是否配套、`CHARSET`/`NUM_CHARS`/`MAX_CHARS_PER_FONT`
   等参数语义是否合理、已有 `data/` 缓存与当前 `CHARSET` 是否一致（发现错误会中断本 Cell 并列出所有问题）
5. 硬件自动调优：按 `AUTO_TUNE`/`TUNE_METHOD` 自动推算 `BATCH_SIZE`/`GEN_BSZ`/`NUM_WORKERS`，
   并打印出来供 Cell 3 使用

**产物：** 输出调优后的 `BATCH_SIZE`、`GEN_BSZ`、`NUM_WORKERS` 三个变量。

> 建议：把整个项目（含本 ipynb）上传/解压后在 Cloud Studio 里**直接打开**，所有目录即可视化，不需要再 clone。

In [ ]:
# ============================================================
# Cell 1 · 环境初始化 + 导入检查 + 硬件自动调优（一般不用改）
# ============================================================
import os, sys, glob, shutil, subprocess, datetime, zipfile, io, base64

# 1) 项目根目录 = 本 ipynb 所在目录（自动探测；不要再 git clone，腾讯云 clone 的目录会不可视化）
def _find_project_root():
    cwd = os.path.abspath("")  # 内核工作目录（可能是 / 或 /workspace）
    markers = ("zi2zi_jit_cloudstudio.ipynb", "lora_single_gpu_finetune_jit.py", "config_font.py")
    cands = [cwd, "/workspace"]
    for _d in sorted(os.listdir("/")):  # 兜底：扫描 / 下所有一级目录
        cands.append(os.path.join("/", _d))
    for _c in cands:
        if os.path.isdir(_c) and any(os.path.exists(os.path.join(_c, m)) for m in markers):
            return os.path.normpath(_c)
    return cwd

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print("[Cell1] 项目根目录:", PROJECT_ROOT)

# 2) 依赖检查（逐模块探测，缺什么补装什么）
# ---------------------------------------------------------------------------
# 国内镜像源（按需切换；某源失败时换下面备选）：
#   pypi 镜像（装除 torch 外的依赖）：
#     腾讯云  https://mirrors.cloud.tencent.com/pypi/simple     （默认，同平台最快）
#     阿里云  https://mirrors.aliyun.com/pypi/simple/
#     清华    https://pypi.tuna.tsinghua.edu.cn/simple
#     中科大  https://pypi.mirrors.ustc.edu.cn/simple/
#     豆瓣    https://pypi.douban.com/simple/
#   torch 专用源（GPU 版；pypi 上的 torch 是 CPU 版，必须走专用源）：
#     官方    https://download.pytorch.org/whl/cu121            （默认，torch 2.5.1 配套）
#     清华    https://mirrors.tuna.tsinghua.edu.cn/pytorch/whl/cu121
# ---------------------------------------------------------------------------
_PYPI_MIRROR = "https://mirrors.cloud.tencent.com/pypi/simple"
_TORCH_INDEX = "https://download.pytorch.org/whl/cu121"
import importlib.util as _ilu

# 依赖清单：pip 包名 -> 实际 import 的模块名（两者不一致的必须显式映射，否则会误判为已装）
_MODULE_MAP = {"opencv-python": "cv2", "fonttools": "fontTools", "Pillow": "PIL",
               "pytorch-msssim": "pytorch_msssim", "torch-fidelity": "torch_fidelity"}
_BASE_DEPS = ["numpy", "opencv-python", "timm", "tensorboard", "scipy", "einops",
              "gdown", "fonttools", "Pillow", "pytorch-msssim", "lpips",
              "torch-fidelity", "tqdm", "matplotlib"]

def _pip(*pkgs, index_url=None, mirror=None):
    cmd = [sys.executable, "-m", "pip", "install", *pkgs]
    if index_url:
        cmd += ["--index-url", index_url]
    elif mirror:
        cmd += ["-i", mirror]
    subprocess.run(cmd, check=True)

def _missing_pkgs():
    """逐模块探测缺失依赖（只 find_spec 不 import，速度快）。返回缺失的 pip 包名。"""
    miss = []
    for pkg in _BASE_DEPS:
        mod = _MODULE_MAP.get(pkg, pkg)
        try:
            if _ilu.find_spec(mod) is None:
                miss.append(pkg)
        except Exception:
            miss.append(pkg)
    return miss

# 1) torch + torchvision：GPU 版必须从 pytorch 专用源装（pypi 上的是 CPU 版），且两者版本需配套
if _ilu.find_spec("torch") is None or _ilu.find_spec("torchvision") is None:
    print("[Cell1] 安装 GPU 版 torch + torchvision（首次约需几分钟）...")
    _pip("torch==2.5.1", "torchvision==0.20.1", index_url=_TORCH_INDEX)

# 2) 其余库：逐模块探测，只补装缺失的（已装的不会重复安装）
_miss = _missing_pkgs()
if _miss:
    print("[Cell1] 补装缺失依赖:", _miss)
    _pip(*_miss, mirror=_PYPI_MIRROR)
else:
    print("[Cell1] 依赖检查通过：所有模块已安装")

import torch
print("[Cell1] torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

# 3) 按 PROJECT_ROOT 重新定位路径（覆盖 Cell 0 中的相对值）
FONTS_DIR   = os.path.join(PROJECT_ROOT, "fonts")
MODELS_DIR  = os.path.join(PROJECT_ROOT, "models")
DATA_DIR    = os.path.join(PROJECT_ROOT, "data")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
EXPORTS_DIR = os.path.join(PROJECT_ROOT, "exports")
FONT_TAG       = os.path.splitext(os.path.basename(TARGET_FONTS[0]))[0]
DATASET_DIR    = os.path.join(DATA_DIR, FONT_TAG, str(CHARSET).lower())
TRAIN_DIR      = os.path.join(DATASET_DIR, "train")
TEST_NPZ_PATH  = os.path.join(DATASET_DIR, "test.npz")
OUTPUT_DIR     = os.path.join(OUTPUTS_DIR, FONT_TAG)
GEN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "generated_chars")
MISSING_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "missing_chars")
for d in (FONTS_DIR, MODELS_DIR, DATA_DIR, OUTPUTS_DIR, EXPORTS_DIR):
    os.makedirs(d, exist_ok=True)

# 4) 导入检查：目标字体 / 源字体 / 预训练模型
for t in TARGET_FONTS:
    p = t if os.path.isabs(t) else os.path.join(PROJECT_ROOT, t)
    assert os.path.exists(p), "[Cell1] 目标字体不存在: %s（请上传到 %s）" % (p, FONTS_DIR)
print("[Cell1] 目标字体 OK:", TARGET_FONTS)

def resolve_source_font():
    if SOURCE_FONT:
        p = SOURCE_FONT if os.path.isabs(SOURCE_FONT) else os.path.join(PROJECT_ROOT, SOURCE_FONT)
        assert os.path.exists(p), "[Cell1] 源字体不存在: " + p
        return p
    target_names = {os.path.basename(t) for t in TARGET_FONTS}
    fonts = []
    for ext in ("*.ttf", "*.otf", "*.ttc"):
        fonts += glob.glob(os.path.join(FONTS_DIR, ext))
    # 也自动搜索 fonts/jigmo/（HanziGen 同款参考字体目录，jigmo.ttf / jigmo2.ttf / jigmo3.ttf）
    jigmo_dir = os.path.join(FONTS_DIR, "jigmo")
    if os.path.isdir(jigmo_dir):
        for ext in ("*.ttf", "*.otf", "*.ttc"):
            fonts += glob.glob(os.path.join(jigmo_dir, ext))
    cand = [f for f in fonts if os.path.basename(f) not in target_names]
    assert cand, "[Cell1] 未找到源字体：请在 fonts/ 放一个参照字体（如 fonts/jigmo/jigmo.ttf），或设置 SOURCE_FONT"
    # jigmo.ttf 覆盖最全，优先；其余按文件名排序
    jigmo_pri = [f for f in cand if os.path.basename(f).lower() in ("jigmo.ttf", "jigmo.otf")]
    return sorted(jigmo_pri or cand)[0]

SOURCE_FONT_PATH = resolve_source_font()
print("[Cell1] 源字体:", SOURCE_FONT_PATH)

ckpt = BASE_CHECKPOINT if os.path.isabs(BASE_CHECKPOINT) else os.path.join(PROJECT_ROOT, BASE_CHECKPOINT)
assert os.path.exists(ckpt), "[Cell1] 预训练模型不存在: %s（请放到 %s）" % (ckpt, MODELS_DIR)
print("[Cell1] 预训练模型:", ckpt)

# 4.5) 配置自检：MODEL<->checkpoint 配套 / 字符集参数 / 数据缓存一致性
import json as _json
import re as _re
from data_processing.charsets import get_charset_codepoints

print("\n[Cell1] ---- 配置自检 ----")
_config_errors = []

def _cok(cond, msg):
    if cond:
        print("  [OK] " + msg)
    else:
        _config_errors.append(msg)
        print("  [ERR] " + msg)

def _cwarn(msg):
    print("  [!!] " + msg)

# a) MODEL <-> BASE_CHECKPOINT 尺寸配套（B-16.pth 配 JiT-B/16，L-16.pth 配 JiT-L/16）
_m_ck = _re.search(r"[_-]([BL])[_-]\d+", os.path.basename(ckpt))
_m_md = _re.search(r"([BL])/", MODEL) if isinstance(MODEL, str) else None
if _m_ck and _m_md:
    _cok(_m_ck.group(1) == _m_md.group(1),
         "checkpoint %s 尺寸=%s 与 MODEL=%s 配套" % (os.path.basename(ckpt), _m_ck.group(1), MODEL))
elif _m_ck and not _m_md:
    _cok(False, "MODEL='%s' 无法解析尺寸（应为 JiT-B/16 或 JiT-L/16）" % MODEL)
else:
    _cwarn("checkpoint 文件名不含 B/L 尺寸标记，跳过配套检查（load_state_dict 严格加载仍会兜底）")

# b) 字符集参数语义
try:
    _cps = get_charset_codepoints(CHARSET)
    _cs = len(_cps)
    print("  [OK] CHARSET=%s 共 %d 字" % (CHARSET, _cs))
except Exception as _e:
    _cps, _cs = None, 0
    _cok(False, "CHARSET='%s' 不受支持: %s" % (CHARSET, _e))

if _cps is not None:
    _cok(NUM_CHARS >= _cs, "NUM_CHARS(%d) >= 字符集大小(%d)" % (NUM_CHARS, _cs))
    if TRAIN_CHARS_PER_FONT > _cs:
        _cwarn("TRAIN_CHARS_PER_FONT(%d) > 字符集大小(%d)，实际会静默取全部 %d 字" % (TRAIN_CHARS_PER_FONT, _cs, _cs))
    if CHARSET == "gbk" and MAX_CHARS_PER_FONT is not None:
        _cwarn("真 GBK 补集建议 MAX_CHARS_PER_FONT=None（当前=%d），否则每字体仅 %d 字参与训练" % (MAX_CHARS_PER_FONT, MAX_CHARS_PER_FONT))
    if MAX_CHARS_PER_FONT is not None and MAX_CHARS_PER_FONT < min(TRAIN_CHARS_PER_FONT, _cs or TRAIN_CHARS_PER_FONT):
        _cwarn("MAX_CHARS_PER_FONT(%d) < TRAIN_CHARS_PER_FONT(%d)：训练时会随机截断到 %d 字，"
               "多生成的图白费、实际训练字数远小于预期（建议设 None）" % (MAX_CHARS_PER_FONT, TRAIN_CHARS_PER_FONT, MAX_CHARS_PER_FONT))

# c) 训练配置一致性
_cok(NUM_FONTS >= len(TARGET_FONTS),
     "NUM_FONTS(%d) >= 目标字体数(%d)" % (NUM_FONTS, len(TARGET_FONTS)))
_cok(RESOLUTION == IMG_SIZE, "RESOLUTION(%d) == IMG_SIZE(%d)" % (RESOLUTION, IMG_SIZE))

# d) 已有数据集缓存与 CHARSET 一致性（改了 CHARSET 后旧数据集不会被自动重建）
if os.path.isdir(TRAIN_DIR) or os.path.exists(TEST_NPZ_PATH):
    _old_cs = set()
    for _mf in glob.glob(os.path.join(TRAIN_DIR, "*", "metadata.json")):
        try:
            with open(_mf, encoding="utf-8") as _f:
                _old_cs.add(str(_json.load(_f).get("charset_filter", "")).lower())
        except Exception:
            pass
    if _old_cs:
        _same = _old_cs == {str(CHARSET).lower()}
        if _same:
            print("  [OK] 已有数据集 charset=%s 与当前 CHARSET=%s 一致" % (sorted(_old_cs), CHARSET))
        elif DO_DATA_PREP:
            # 要重建数据集，charset 不一致不算错误（重建后就一致了），仅提示
            _cwarn("已有数据集 charset=%s 与当前 CHARSET=%s 不一致；DO_DATA_PREP=True 将重建数据集" % (sorted(_old_cs), CHARSET))
        else:
            # 不重建却 charset 不一致 → 会误用旧数据集训练，必须报错拦截
            _cok(False, "已有数据集 charset=%s 与当前 CHARSET=%s 不一致，且 DO_DATA_PREP=False 会误用旧数据集（请把 DO_DATA_PREP 设为 True 重建）" % (sorted(_old_cs), CHARSET))
    else:
        _cwarn("已有数据集但读不到 metadata.json（可能是旧版本），建议删除 %s 后重跑" % DATASET_DIR)
else:
    print("  [OK] 无已有数据集（首次运行）")

# e) 汇总：有错误则中断本 Cell
if _config_errors:
    print("\n[Cell1] !!! 配置自检发现 %d 个错误，请修正 Cell 0 后重跑本 Cell !!!" % len(_config_errors))
    for _i, _e in enumerate(_config_errors, 1):
        print("  %d) %s" % (_i, _e))
    raise RuntimeError("配置自检未通过：%d 个错误" % len(_config_errors))
print("[Cell1] 配置自检全部通过")
print()

# 5) 硬件自动调优：自动推算 BATCH_SIZE / GEN_BSZ / NUM_WORKERS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# V100 检测：老卡（cc<8）无 bf16 张量核，torch.compile 的 Inductor 无法原生编译 bf16
# kernel（只会在推理时反复 "skipping" 刷屏）。据此标记 IS_V100，供 Cell3/Cell5 决定
# 是否禁用 compile / 切换 V100 专用推理脚本。
IS_V100 = False
if device.type == "cuda":
    try:
        if torch.cuda.get_device_capability(device)[0] < 8:
            IS_V100 = True
    except Exception:
        pass
print("[Cell1] V100 检测:", "是（cc<8，将禁用 compile / 切换 V100 推理脚本）" if IS_V100 else "否")

try:
    from util.auto_tune import auto_tune, probe_batch_size  # noqa
    HAS_AUTO_TUNE = True
except ImportError:
    HAS_AUTO_TUNE = False
    def auto_tune(**kw):
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
        per = 0.25 if kw.get("model_name", "JiT-B/16") == "JiT-B/16" else 0.5
        bsz = int(max(total * kw.get("reserve", 0.85) - 1.5, 0) / per) if total else kw.get("batch_size_fallback", 16)
        bsz = max(1, min(bsz, kw.get("max_batch", 128)))
        return {"batch_size": bsz,
                "gen_bsz": max(1, min(bsz, kw.get("max_gen_bsz", 32))),
                "num_workers": min(os.cpu_count() or 1, kw.get("num_workers_cap", 12)),
                "method": "fallback"}

if AUTO_TUNE:
    TUNE = auto_tune(method=TUNE_METHOD, model_name=MODEL, img_size=IMG_SIZE,
                     reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH,
                     max_gen_bsz=TUNE_MAX_GEN_BSZ, num_workers_cap=TUNE_NUM_WORKERS_CAP,
                     batch_size_fallback=BATCH_SIZE)
    BATCH_SIZE = TUNE["batch_size"]
    GEN_BSZ    = TUNE["gen_bsz"]
    NUM_WORKERS = TUNE["num_workers"]
    print("[Cell1] 自动调优完成（%s）-> batch_size:%d gen_bsz:%d num_workers:%d"
          % (TUNE.get("method", "?"), BATCH_SIZE, GEN_BSZ, NUM_WORKERS))
else:
    NUM_WORKERS = min(os.cpu_count() or 1, TUNE_NUM_WORKERS_CAP)
    print("[Cell1] AUTO_TUNE=False，使用固定 batch_size:", BATCH_SIZE)

print("[Cell1] 就绪，继续执行 Cell 2（数据准备）")

## Cell 2 · 数据准备（字体 -> `train/` `test/` `test.npz`）

**🖥️ 推荐配置：CPU 进阶级 16核32G（省钱备选 8核16G）**

- 渲染字形 + 海量小文件写入，**纯 CPU**，瓶颈在内存缓存与磁盘 I/O（`NUM_WORKERS_DATA_PREP` 并行时吃多核）。
- 字符集越大越久：gb2312 约几分钟，真 GBK 全量 2 万字约 10~30 分钟。32G 内存可避免海量小文件触发 Swap。
- **不用开 GPU**（开 GPU 也不加速，反而白花机时）。

**本 Cell 做什么（无需修改）：**

- 只把 `TARGET_FONTS` 指定的字体放进临时目录，调用 `scripts/generate_font_dataset.py`
- 渲染字形、按 `CHARSET` 过滤、抽训练/测试字、生成 `data/<字体>/` 下的数据集
- 已在已有数据集时自动跳过

**产物：** `data/<字体>/train/`（训练图）、`data/<字体>/test/`（测试图）、`data/<字体>/test.npz`（推理用）。

In [ ]:
# ============================================================
# Cell 2 · 数据准备（一般不用改）
# ============================================================
if DO_DATA_PREP and CLEAR_OTHER_FONTS_DATA and os.path.isdir(DATA_DIR):
    # 训练新字体时清掉其他字体的数据集（默认删除省磁盘；设 False 可保留，换回旧字体免重新生成）
    _keep = {FONT_TAG, "_staging_fonts"}
    for _d in sorted(os.listdir(DATA_DIR)):
        _p = os.path.join(DATA_DIR, _d)
        if _d in _keep or not os.path.isdir(_p):
            continue
        shutil.rmtree(_p, ignore_errors=True)
        print("[Cell2] 已清理其他字体数据集:", _p)

if DO_DATA_PREP:
    if os.path.exists(TRAIN_DIR) and os.path.exists(TEST_NPZ_PATH):
        print("[Cell2] 数据集已存在，跳过生成:", DATASET_DIR)
    else:
        staging = os.path.join(DATA_DIR, "_staging_fonts")
        os.makedirs(staging, exist_ok=True)
        for t in TARGET_FONTS:
            p = t if os.path.isabs(t) else os.path.join(PROJECT_ROOT, t)
            shutil.copy2(p, staging)
        cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", "generate_font_dataset.py"),
               "--source-font", SOURCE_FONT_PATH,
               "--font-dir", staging,
               "--output-dir", DATASET_DIR,
               "--train-chars-per-font", str(TRAIN_CHARS_PER_FONT),
               "--test-chars-per-font", str(TEST_CHARS_PER_FONT),
               "--resolution", str(RESOLUTION),
               "--charset", CHARSET,
               "--train-seed", str(TRAIN_SEED),
               "--test-seed", str(TEST_SEED),
               "--num-workers", str(NUM_WORKERS_DATA_PREP)]
        print("[Cell2] 命令:", " ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print("[Cell2] DO_DATA_PREP=False，跳过")

## Cell 3 · LoRA 训练

**🖥️ 推荐配置：V100（性能首选）**

- 阶段二"训练模型"，EDM 扩散 + LoRA 反向传播 + 在线 FID，**全程吃 GPU**，是整套流程最烧机时的一步。
- 文档推荐 **V100**（高带宽保证数据"秒进秒出"，显卡满负荷）；预算受限可用 **T4**（16G 够跑，但明显更慢）；**A10** 是兼顾速度与成本的备选。
- `AUTO_TUNE=True` 会自动按显存定 batch，无需手动猜。

**本 Cell 做什么（无需修改）：**

- 用 Cell 0 的超参调用 `lora_single_gpu_finetune_jit.py`
- `AUTO_TUNE=True` 且 `TUNE_METHOD=probe` 时：会先构建一个与真实训练**完全一致**的模型
  （含 LoRA 注入），实测单样本显存，再精调 `batch_size` ——
  **改 `LORA_R`/模型变体/分辨率后都不用再手动猜显存**
- 训练中按 `EVAL_FREQ` 在线生成样例图（`outputs/<字体>/`）

**产物：** `outputs/<字体>/checkpoint-last.pth`（LoRA checkpoint，后续 Cell 都用它）。

> 训练轮数、CFG 等调整建议见 Cell 0 上方表格。

In [ ]:
# ============================================================
# Cell 3 · LoRA 训练（一般不用改）
# ============================================================
if DO_TRAIN:
    import torch
    # ---- 训练前显存清理（修复 Interrupt 只停执行、不释放显存导致下次训练变慢）----
    # Interrupt 后训练循环里的变量会被异常 traceback 持有引用，显存不会自动释放，
    # 导致下次 probe 低估可用显存、把 batch 压到 1。这里主动清理：
    # ① 清掉 sys.last_traceback 等异常引用 ② 删除残留的模型/优化器变量 ③ 回收 CUDA 缓存
    import gc
    _cleared = []
    for _attr in ("last_traceback", "last_value", "last_type"):
        if hasattr(sys, _attr):
            try:
                delattr(sys, _attr)
                _cleared.append("sys." + _attr)
            except Exception:
                pass
    for _k in ("model", "model_without_ddp", "optimizer", "data_loader_train",
               "probe_model", "loss_scaler", "ck", "sd", "_ck", "_sd"):
        if _k in globals():
            try:
                del globals()[_k]
                _cleared.append(_k)
            except Exception:
                pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info()
        print("[Cell3] 显存清理完成（清除: %s）→ 可用 %.2f GB / 共 %.2f GB"
              % (", ".join(_cleared) or "无残留", _free / 1024**3, _total / 1024**3))
    from lora_single_gpu_finetune_jit import get_args_parser, main as lora_main

    argv = [
        "--data_path", TRAIN_DIR,
        "--test_npz_path", TEST_NPZ_PATH,
        "--output_dir", OUTPUT_DIR,
        "--base_checkpoint", BASE_CHECKPOINT,
        "--model", MODEL,
        "--img_size", str(IMG_SIZE),
        "--num_fonts", str(NUM_FONTS),
        "--num_chars", str(NUM_CHARS),
        "--lora_r", str(LORA_R),
        "--lora_alpha", str(LORA_ALPHA),
        "--lora_targets", LORA_TARGETS,
        "--lora_dropout", str(LORA_DROPOUT),
        "--proj_dropout", str(PROJ_DROPOUT),
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--blr", str(BLR),
        "--min_lr", str(MIN_LR),
        "--warmup_epochs", str(WARMUP_EPOCHS),
        "--save_last_freq", str(SAVE_LAST_FREQ),
        "--P_mean", str(P_MEAN),
        "--P_std", str(P_STD),
        "--noise_scale", str(NOISE_SCALE),
        "--cfg", str(CFG),
        "--num_images", str(NUM_IMAGES),
        "--seed", str(SEED),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    if MAX_CHARS_PER_FONT is not None:
        argv += ["--max_chars_per_font", str(MAX_CHARS_PER_FONT)]
    if CKPT_BACKUP_EVERY:
        argv += ["--ckpt_backup_every", str(CKPT_BACKUP_EVERY),
                 "--ckpt_backup_keep", str(CKPT_BACKUP_KEEP)]

    args = get_args_parser().parse_args(argv)
    if ONLINE_EVAL:
        args.online_eval = True
    if EVAL_STEP_FOLDERS:
        args.eval_step_folders = True
    # 修正：Cell 0 的 EVAL_FREQ / GEN_BSZ 此前没传给 args，实际一直用 argparse 默认值
    # （EVAL_FREQ 默认 40），导致明明设了 10 却每 40 轮才出一组评估图。这里补上赋值。
    args.eval_freq = EVAL_FREQ
    args.gen_bsz = GEN_BSZ          # probe 成功时会被实测值覆盖，仅作 AUTO_TUNE 关闭时的兜底
    args.num_workers = NUM_WORKERS

    # probe 实测精调 batch_size（自动反映 LoRA / encoder / 模型变体对显存的影响）
    if AUTO_TUNE and TUNE_METHOD == "probe" and torch.cuda.is_available() and HAS_AUTO_TUNE:
        # probe 前先查 GPU 是否已被占用：残留显存会让 probe 低估可用空间，把 batch 压到 1
        _free, _total = torch.cuda.mem_get_info()
        _used_gb = (_total - _free) / 1024 ** 3
        print("[Cell3] probe 前 GPU 显存：已用 %.2f GB / 共 %.2f GB" % (_used_gb, _total / 1024 ** 3))
        if _used_gb > 3.0:
            print("[Cell3] 警告：GPU 已占用 %.2f GB，疑似上次训练未释放（Interrupt 只停执行，不释放显存）" % _used_gb)
            print("[Cell3]       probe 将低估可用显存并把 batch 压到 1，请先 Restart Kernel 再重跑本 Cell")
        print("[Cell3] 构建探测模型（与真实训练一致，含 LoRA 注入）实测单样本显存 ...")
        import torch._dynamo
        torch._dynamo.config.cache_size_limit = 128
        from denoiser import Denoiser
        from util.lora_utils import (inject_lora, mark_only_lora_as_trainable,
                                     _is_lora_state_dict, resolve_checkpoint_path)

        probe_model = Denoiser(args)
        probe_model.update_ema = lambda: None
        ckpt_path = resolve_checkpoint_path(args.base_checkpoint)
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
        is_lora = _is_lora_state_dict(sd)
        del ck
        if not is_lora:
            probe_model.load_state_dict(sd, strict=True)
        targets = [t.strip() for t in args.lora_targets.split(",") if t.strip()]
        inject_lora(probe_model.net, targets, r=args.lora_r, alpha=args.lora_alpha, dropout=args.lora_dropout)
        if is_lora:
            probe_model.load_state_dict(sd, strict=True)
        mark_only_lora_as_trainable(probe_model, train_font_emb=True)
        probe_model.to(device)

        safe, per_gb = probe_batch_size(probe_model, device, img_size=IMG_SIZE,
                                        num_fonts=NUM_FONTS, num_chars=NUM_CHARS,
                                        reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH)
        if safe is not None:
            print("[Cell3] 实测单样本显存: %.3f GB -> batch_size=%d" % (per_gb, safe))
            args.batch_size = safe
            args.gen_bsz = max(1, min(safe, TUNE_MAX_GEN_BSZ))
        del probe_model
        torch.cuda.empty_cache()

    print("[Cell3] 最终 batch_size:", args.batch_size, "| gen_bsz:", args.gen_bsz, "| num_workers:", args.num_workers)
    # ---- 训练前 checkpoint 健康检查（防权重损坏导致的 loss=NaN） ----
    import os
    from util.lora_utils import resolve_checkpoint_path
    _ck_path = resolve_checkpoint_path(args.base_checkpoint) if args.base_checkpoint else None
    if _ck_path and os.path.exists(_ck_path):
        _ck = torch.load(_ck_path, map_location='cpu', weights_only=False)
        _sd = _ck['model'] if isinstance(_ck, dict) and 'model' in _ck else _ck
        _bad = [k for k, v in _sd.items() if v.is_floating_point() and (torch.isnan(v).any() or torch.isinf(v).any())]
        if _bad:
            raise RuntimeError('[Cell3] base_checkpoint 含 NaN/Inf 权重: %s，请重新下载该模型文件' % _bad[:8])
        _maxw = max((float(v.abs().max()) for k, v in _sd.items() if v.is_floating_point()), default=0.0)
        print('[Cell3] checkpoint 健康检查通过：0 个 NaN/Inf 权重，最大 |权重|=%.3e' % _maxw)
        del _ck, _sd, _bad
    else:
        print('[Cell3] 未找到 base_checkpoint，跳过健康检查')

    # ---- 断点续训：检测 checkpoint-last 并校验配置，一致才从中断处继续 ----
    _resume_ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    if os.path.exists(_resume_ckpt):
        try:
            _ck = torch.load(_resume_ckpt, map_location='cpu', weights_only=False)
            _saved = _ck.get('args') if isinstance(_ck, dict) else None
            # 参与一致性校验的关键训练配置
            # （epochs 不校验：改总轮数=延长训练，是最典型的续训场景，不影响模型结构；
            #   batch_size 不校验：probe 每次可能微调，不影响模型结构）
            _fields = ['data_path', 'model', 'img_size', 'num_fonts', 'num_chars',
                       'max_chars_per_font', 'lora_r', 'lora_alpha', 'lora_targets',
                       'lora_dropout', 'proj_dropout', 'blr', 'noise_scale']
            _diff = []
            if _saved is not None:
                for _f in _fields:
                    _a = getattr(_saved, _f, None)
                    _b = getattr(args, _f, None)
                    if _a != _b:
                        _diff.append('%s: %r != %r' % (_f, _a, _b))
            _saved_epoch = _ck.get('epoch') if isinstance(_ck, dict) else None
            # epochs 变化提示：影响取决于 lr_schedule（constant 时 LR 恒定，改轮数不影响 LR）
            _saved_epochs = getattr(_saved, 'epochs', None) if _saved is not None else None
            if _saved_epochs is not None and _saved_epochs != args.epochs:
                if getattr(args, 'lr_schedule', '') == 'constant':
                    _lr_note = 'LR 为 constant 调度，恒定不受影响'
                else:
                    _lr_note = 'LR 按新总轮数重算、曲线不连续（等同 warm restart）'
                print('[Cell3] EPOCHS %r -> %r：本次为延长训练（续训），%s' % (_saved_epochs, args.epochs, _lr_note))
            if _diff:
                print('[Cell3] 检测到 checkpoint-last 但配置与上次不一致，从头训练：')
                for _d in _diff:
                    print('   !! ' + _d)
            elif isinstance(_saved_epoch, int) and _saved_epoch < args.epochs - 1:
                args.resume = _resume_ckpt
                print('[Cell3] 断点续训：从 epoch %d 继续（checkpoint-last）' % (_saved_epoch + 1))
            else:
                print('[Cell3] checkpoint-last 已完成全部 epoch（%d/%d），无需重训' % (_saved_epoch or 0, args.epochs))
            del _ck
        except Exception as _e:
            print('[Cell3] checkpoint-last 读取失败（%s），从头训练' % _e)
    else:
        print('[Cell3] 未找到 checkpoint-last，从头训练')

    lora_main(args)
else:
    print("[Cell3] DO_TRAIN=False，跳过")

## Cell 4 · 推理生成 PNG（测试集字）

**🖥️ 推荐配置：V100（GPU）**

- 扩散采样必须 GPU。仅生成 test 集字（默认 `NUM_IMAGES` 个，量很小），通常几分钟内跑完，GPU 机时占比小。
- 想更省可在跑完 Cell 3 后**切回 CPU 高配**再跑本 Cell 吗？——不行，采样必须 GPU，但量小，直接和 Cell 3 一起在 V100 上跑即可。

**本 Cell 做什么（无需修改）：**

- 用 `outputs/<字体>/checkpoint-last.pth` + `data/<字体>/test.npz` 调用 `generate_chars.py`
- 生成 test 集每个字的单字图；仅当 `GENERATE_PAIRWISE` 非 None 时才额外输出对比图（`"src_gen"`=源字形|生成结果、`"target_gen"`=目标字形|生成结果，写入 `compare/` 子目录供指标计算用）
- `AUTO_TUNE=True` 时会按显存自动收紧推理批量

**产物：** `outputs/<字体>/generated_chars/`（`compare/` 对比图 + `generated/` 单字图）。

> 只针对"目标字体本来就有、且测试集抽中的字"。想补"目标字体缺失的字"请看 **Cell 5**。

In [ ]:
# ============================================================
# Cell 4 · 推理生成（一般不用改）
# ============================================================
if DO_GENERATE:
    ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    assert os.path.exists(ckpt), "[Cell4] 未找到 checkpoint: %s" % ckpt

    # V100（cc<8）禁用 compile：切换到 generate_chars_V100，避免 Inductor 刷屏警告
    if IS_V100:
        from generate_chars_V100 import get_args_parser, main as gen_main
    else:
        from generate_chars import get_args_parser, main as gen_main
    args = get_args_parser().parse_args([
        "--checkpoint", ckpt,
        "--test_npz", TEST_NPZ_PATH,
        "--output_dir", GEN_OUTPUT_DIR,
        "--device", "auto",
    ])
    if GENERATE_NUM_IMAGES is not None:
        args.num_images = GENERATE_NUM_IMAGES
    if GENERATE_BATCH_SIZE is not None:
        args.batch_size = GENERATE_BATCH_SIZE
    if GENERATE_CFG is not None:
        args.cfg = GENERATE_CFG
    if GENERATE_SAMPLING_METHOD is not None:
        args.sampling_method = GENERATE_SAMPLING_METHOD
    if GENERATE_NUM_SAMPLING_STEPS is not None:
        args.num_sampling_steps = GENERATE_NUM_SAMPLING_STEPS
    if GENERATE_PAIRWISE is not None:
        args.pairwise = GENERATE_PAIRWISE

    if AUTO_TUNE and torch.cuda.is_available():
        try:
            from util.auto_tune import auto_tune
            args.batch_size = auto_tune(method="table", model_name=MODEL, img_size=IMG_SIZE,
                                        reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH,
                                        max_gen_bsz=TUNE_MAX_GEN_BSZ,
                                        batch_size_fallback=args.batch_size,
                                        verbose=False)["gen_bsz"]
        except Exception:
            pass

    print("[Cell4] 推理生成中 ... batch_size:", args.batch_size)
    gen_main(args)
else:
    print("[Cell4] DO_GENERATE=False，跳过")

## Cell 5 · 缺失字补集生成（类似 HanziGen 的缺字补全）

**🖥️ 推荐配置：V100 GPU（吃 GPU 计算）+ 尽量少写盘**

- 扩散采样必须 GPU，且**生成全部缺失字**（`MISSING_NUM_IMAGES=None` 时，gbk 可多达数千字），量级远超 Cell 4。
- ⚠️ **为什么它比 Cell 3/4 都慢**：GPU 计算本身不多，但**每个字要写 1~2 个 PNG**（`generated/` + `compare/`），几千字就是几千次同步磁盘写，I/O 成了瓶颈 → **GPU 确实没吃满**。
- **提速办法**（见 Cell 0）：
  - `MISSING_PAIRWISE=""`：不写对比图，省一半 I/O；
  - `MISSING_BATCH_SIZE` 调大（V100 可设 128+）：减少 Python 循环与 `.to()`/autocast 开销；
  - 若只是先看效果，`MISSING_NUM_IMAGES=100` 先生成少量，确认满意再全量。
- ✅ **断点续传**：已支持。中途中止后重跑本 Cell，会扫描 `missing_chars/generated/` 里已生成的 PNG 并**自动跳过**，只算剩余缺失字，不重复烧机时。

**本 Cell 做什么（无需修改）：**

1. 用 fontTools 读取目标字体的 cmap，计算 **CHARSET 字符集 − 目标字体已覆盖字符 = 缺失字**
2. 缺失字须能被参照字体渲染（模型以参照字形为 content 输入）——所以对比图里的"源字形"来自**参照字体**，不是目标字体有的字
3. 样式参考图来自**目标字体自身**（与训练时的 ref 网格一致）
4. 用训练好的 checkpoint 逐个生成缺失字 PNG，并输出 `missing_chars.txt` 清单

**产物：** `outputs/<字体>/missing_chars/`

- `generated/U+XXXX.png`：补全的缺失字
- `compare/U+XXXX.png`：`源字形 | 生成结果` 对比图（仅 `MISSING_PAIRWISE="src_gen"` 时生成）
- `missing_chars.txt`：缺失字清单（U+XXXX 与字符对照）

**`MISSING_*` 参数说明（详见 Cell 0）：**

- `MISSING_PAIRWISE` 三档：`"src_gen"`（默认）= 输出 `源字形 | 生成结果` 左右对比图，写盘量翻倍；`""` = 只出生成图，省一半 I/O、显著提速；`target_gen` 在补集场景**不可用**——目标字体没有这些缺失字，脚本未实现该分支。
- `MISSING_REF_CHARS`：逗号分隔的样式参考字，留空则自动从目标字体可渲染的字里随机挑（脚本默认挑 8 个，上限 8）。
- `MISSING_CFG` / `MISSING_SAMPLING_METHOD` / `MISSING_NUM_SAMPLING_STEPS`：采样参数，`None` = 沿用训练 checkpoint 里的配置（同 Cell 4 规则）。
- `MISSING_NUM_IMAGES`：`None` = 生成全部缺失字（gbk 可多达数千字）；先设小值（如 100）可快速看效果，确认后再全量。

> 用法示例：字体缺某些简/繁体字时一键补齐。**提示**：想让字符标签与预训练对齐、补集效果最好，
> 训练时建议 `TRAIN_CHARS_PER_FONT` 覆盖整个 CHARSET、`MAX_CHARS_PER_FONT=None`。

In [ ]:
# ============================================================
# Cell 5 · 缺失字补集（一般不用改）
# ============================================================
if DO_MISSING_GEN:
    ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    assert os.path.exists(ckpt), "[Cell5] 未找到 checkpoint: %s" % ckpt

    target = TARGET_FONTS[0]
    target_path = target if os.path.isabs(target) else os.path.join(PROJECT_ROOT, target)

    cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", "generate_missing_chars.py"),
           "--checkpoint", ckpt,
           "--target-font", target_path,
           "--source-font", SOURCE_FONT_PATH,
           "--charset", MISSING_CHARSET or CHARSET,
           "--output-dir", MISSING_OUTPUT_DIR,
           "--batch-size", str(MISSING_BATCH_SIZE or 32),
           "--pairwise", MISSING_PAIRWISE or "none",
           "--seed", str(SEED),
           "--device", "auto"]
    if MISSING_NUM_IMAGES is not None:
        cmd += ["--num-images", str(MISSING_NUM_IMAGES)]
    if MISSING_CFG is not None:
        cmd += ["--cfg", str(MISSING_CFG)]
    if MISSING_SAMPLING_METHOD is not None:
        cmd += ["--sampling-method", MISSING_SAMPLING_METHOD]
    if MISSING_NUM_SAMPLING_STEPS is not None:
        cmd += ["--num-sampling-steps", str(MISSING_NUM_SAMPLING_STEPS)]
    if MISSING_REF_CHARS:
        cmd += ["--ref-chars", MISSING_REF_CHARS]

    print("[Cell5] 命令:", " ".join(cmd))
    # V100（cc<8）不支持 bf16 原生编译：通过环境变量让补集脚本切换到
    # generate_chars_V100.py（禁用 torch.compile），消除 Inductor 刷屏警告。
    env = dict(os.environ)
    if IS_V100:
        env["ZI2ZI_V100"] = "1"
        print("[Cell5] 检测到 V100，使用 generate_chars_V100.py（禁用 compile）")
    subprocess.run(cmd, check=True, env=env)
else:
    print("[Cell5] DO_MISSING_GEN=False，跳过")

## Cell 5b · 指定字符生成（可选 · 归类在 Cell 5 下）

训练完成后，如果你只想补**指定的一批字**（而不是按 CHARSET 自动补全部缺失字），
或想对**同一个字生成多张**（扩散每次采样结果不同，可挑最好的一张），用本 Cell。

**与 Cell 5 的区别：**

- Cell 5 = 自动补全「字符集 − 目标字体已覆盖」的**全部**缺失字，每个字只出 1 张；
- 本 Cell = **你指定**要生成哪些字（不管目标字体缺不缺），且每个字可生成 N 张。

**前置条件：** 已完成训练（`outputs/<字体>/checkpoint-last.pth` 存在）。
本 Cell 是**纯推理**，改下面任何参数都**不需要重新训练、不需要删 checkpoint**。

**快速设置区（在下方代码 Cell 顶部）：**

- `DO_PICK_CHARS`：是否运行本 Cell（默认 `False`，需要时改 `True`）
- `PICK_CHARS`：要生成的字符（可多个，如 `"国"`、`"国家汉"`）
- `PER_CHAR_IMAGES`：每个字符生成几张图（扩散采样随机，多张可挑最好的）
- `PICK_CFG` / `PICK_SAMPLING_METHOD` / `PICK_NUM_SAMPLING_STEPS`：采样参数，`None`=沿用 checkpoint
- `PICK_REF_CHARS`：样式参考字（留空自动从目标字体可渲染的字里挑）
- `PICK_PAIRWISE`：`"src_gen"`=额外输出 源字形|生成结果 对比图；`""`=只出生成图

**产物：** `outputs/<字体>/picked_chars/`

- `generated/U+XXXX_KK.png`：每个字符的第 K 张生成图
- `compare/U+XXXX_KK.png`：对比图（仅 `PICK_PAIRWISE="src_gen"` 时）

> 提示："源字形"（content）来自**参照字体**；"样式参考"来自**目标字体**自身（与训练一致）。
> 若某字符参照字体也无法渲染，会被自动跳过并提示。

In [ ]:
# ============================================================
# Cell 5b · 指定字符生成（可选，不重新训练）
# ============================================================

# ---------- 快速设置区（只改这里） ----------
DO_PICK_CHARS         = False      # 是否运行本 Cell（需要时改 True）
PICK_CHARS            = "国"       # 要生成的字符（可多个，如 "国家汉"）
PER_CHAR_IMAGES       = 5          # 每个字符生成几张（扩散每次采样结果不同）
PICK_CFG              = None       # 引导强度（None=沿用 checkpoint）
PICK_SAMPLING_METHOD  = None       # euler / heun / ab2（None=沿用 checkpoint）
PICK_NUM_SAMPLING_STEPS = None     # 采样步数（None=沿用方法默认）
PICK_REF_CHARS        = ""         # 逗号分隔的样式参考字（留空自动挑）
PICK_PAIRWISE         = "src_gen"  # "src_gen"=输出 源字形|生成结果 对比图；""=只出生成图

if DO_PICK_CHARS:
    import torch, cv2, numpy as np
    from PIL import Image
    from contextlib import nullcontext
    from denoiser import Denoiser
    from util.lora_utils import _is_lora_state_dict, inject_lora
    from util.misc import get_amp_dtype
    # V100（cc<8）禁用 compile：切换到 generate_chars_V100，避免 Inductor 刷屏警告
    if IS_V100:
        from generate_chars_V100 import DEFAULT_STEPS_BY_METHOD, patch_torch_for_device, resolve_device
    else:
        from generate_chars import DEFAULT_STEPS_BY_METHOD, patch_torch_for_device, resolve_device
    from data_processing.charsets import get_charset_codepoints
    from data_processing.font_utils import GlyphRenderer, get_cjk_codepoints, load_font
    from data_processing.pipeline import create_combined_image, create_reference_grid, _extract_ref

    ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    assert os.path.exists(ckpt), "[Cell5b] 未找到 checkpoint: %s" % ckpt

    device = resolve_device("auto")
    patch_torch_for_device(device)
    use_cuda_amp = device.type == "cuda"

    # ---- 1) 加载 checkpoint + 模型（与 generate_missing_chars.py 一致） ----
    print("[Cell5b] 加载 checkpoint:", ckpt)
    checkpoint = torch.load(ckpt, map_location="cpu", weights_only=False)
    ckpt_args = checkpoint["args"]
    model = Denoiser(ckpt_args)
    state_dict = checkpoint.get("model_ema1", checkpoint.get("model", checkpoint))
    is_lora = _is_lora_state_dict(state_dict)
    if is_lora:
        r = getattr(ckpt_args, "lora_r", 8)
        a = getattr(ckpt_args, "lora_alpha", 16)
        d = getattr(ckpt_args, "lora_dropout", 0.0)
        tgt = [t.strip() for t in getattr(ckpt_args, "lora_targets", "qkv,proj,w12,w3").split(",") if t.strip()]
        inject_lora(model.net, tgt, r=r, alpha=a, dropout=d)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()

    # ---- 2) 采样参数：Cell 内覆盖 > checkpoint > 默认 ----
    method = PICK_SAMPLING_METHOD or getattr(ckpt_args, "sampling_method", "heun")
    cfg = PICK_CFG if PICK_CFG is not None else getattr(ckpt_args, "cfg", 4.0)
    steps = PICK_NUM_SAMPLING_STEPS or DEFAULT_STEPS_BY_METHOD[method]
    model.cfg_scale = cfg
    model.steps = steps
    model.method = method
    model.cfg_interval = (getattr(ckpt_args, "interval_min", 0.0), getattr(ckpt_args, "interval_max", 1.0))
    print("[Cell5b] 采样: %s steps=%d cfg=%.2f" % (method, steps, cfg))

    # ---- 3) 解析指定字符 ----
    wanted = [ord(ch) for ch in PICK_CHARS if ch.strip()]
    print("[Cell5b] 待生成字符: %s 共 %d 字，每字 %d 张" % (PICK_CHARS, len(wanted), PER_CHAR_IMAGES))

    # ---- 4) 字体渲染器（content 来自参照字体，样式参考来自目标字体） ----
    target = TARGET_FONTS[0]
    target_path = target if os.path.isabs(target) else os.path.join(PROJECT_ROOT, target)
    src_renderer = GlyphRenderer(SOURCE_FONT_PATH, 256)
    tgt_renderer = GlyphRenderer(target_path, 256)

    # ---- 5) 构造样式参考网格（来自目标字体可渲染的字，与训练 ref 网格一致） ----
    target_font, _ = load_font(target_path)
    target_cps = get_cjk_codepoints(target_font)
    charset_cps = get_charset_codepoints(CHARSET)
    ref_pool = sorted(target_cps & charset_cps)
    if PICK_REF_CHARS:
        refs = [ord(ch) for ch in PICK_REF_CHARS if ch.strip() and ord(ch) in target_cps][:8]
    else:
        refs = []
    if len(refs) < 8:
        import random as _r
        _rng = _r.Random(SEED)
        refs = _rng.sample(ref_pool, min(len(ref_pool), 8))
    refs = refs[:8]
    print("[Cell5b] 样式参考字:", "".join(chr(c) for c in refs))

    white = Image.new("RGB", (256, 256), (255, 255, 255))
    g1 = create_reference_grid(tgt_renderer, refs[:4])
    g2 = create_reference_grid(tgt_renderer, refs[4:]) if len(refs) > 4 else None
    combined = create_combined_image(white, white, g1, g2 if g2 is not None else g1)
    ref_img = _extract_ref(combined, 0, 128)
    ref_arr = np.array(ref_img).transpose(2, 0, 1)

    # ---- 6) 字符标签映射（与训练 charset_index 一致） ----
    index_map = {cp: i for i, cp in enumerate(sorted(charset_cps))}
    num_chars = getattr(ckpt_args, "num_chars", None)

    out_dir = os.path.join(OUTPUT_DIR, "picked_chars")
    gen_folder = os.path.join(out_dir, "generated")
    cmp_folder = os.path.join(out_dir, "compare") if PICK_PAIRWISE == "src_gen" else None
    os.makedirs(gen_folder, exist_ok=True)
    if cmp_folder:
        os.makedirs(cmp_folder, exist_ok=True)

    # ---- 7) 逐个字符、逐张生成 ----
    for cp in wanted:
        if num_chars and index_map.get(cp, 0) >= num_chars:
            print("[Cell5b] 跳过 %s：超出字符嵌入空间(num_chars=%d)" % (chr(cp), num_chars))
            continue
        content = src_renderer.render(cp)
        if content is None:
            print("[Cell5b] 跳过 %s：参照字体无法渲染" % chr(cp))
            continue
        content_img = np.array(content).transpose(2, 0, 1)
        char_label = index_map.get(cp, 0)
        for k in range(PER_CHAR_IMAGES):
            font_b = torch.tensor([0], dtype=torch.long, device=device)
            char_b = torch.tensor([char_label], dtype=torch.long, device=device)
            style_b = torch.from_numpy(ref_arr[None].copy()).float().to(device) / 255.0 * 2.0 - 1.0
            content_b = torch.from_numpy(content_img[None].copy()).float().to(device) / 255.0 * 2.0 - 1.0
            labels = (font_b, char_b, style_b, content_b)
            with (torch.amp.autocast("cuda", dtype=get_amp_dtype()) if use_cuda_amp else nullcontext()):
                generated = model.generate(labels)
            generated = (generated + 1) / 2
            gen_img = np.round(np.clip(generated[0].detach().cpu().numpy().transpose([1, 2, 0]) * 255, 0, 255))
            gen_img = gen_img.astype(np.uint8)[:, :, ::-1]
            name = "U+%04X_%02d" % (cp, k + 1)
            cv2.imwrite(os.path.join(gen_folder, name + ".png"), gen_img)
            if cmp_folder:
                src_img = content_img.transpose([1, 2, 0])[:, :, ::-1]
                cv2.imwrite(os.path.join(cmp_folder, name + ".png"), np.concatenate([src_img, gen_img], axis=1))
            print("[Cell5b] 生成 %s 第 %d/%d 张 -> %s.png" % (chr(cp), k + 1, PER_CHAR_IMAGES, name))
    print("[Cell5b] 完成！输出目录:", out_dir)
else:
    print("[Cell5b] DO_PICK_CHARS=False，跳过")

## Cell 6 · 导出打包 + 浏览器下载

**🖥️ 推荐配置：CPU 进阶级 16核32G（首选）或 8核16G（省钱备选）**

- 阶段三"生成与导出"，zip 打包 + base64，**纯 CPU**，吃内存缓存（海量小文件频繁读写）。
- 文档明确：**不要用 4核8G 及以下**——低内存会导致磁盘 Swap 甚至崩溃，导出失败。
- **不用开 GPU**。

**本 Cell 做什么（无需修改）：**

- 把 Cell 4 的 `generated_chars/` 与 Cell 5 的 `missing_chars/` 下所有 PNG
  （可选含 LoRA checkpoint）打包成 zip
- 保存到 `exports/`（可视化目录，可在左侧文件树下载），并给出**浏览器一键下载**按钮
- zip 超过 1 GB 时不再生成 base64 下载链接，请直接从文件树下载

In [ ]:
# ============================================================
# Cell 6 · 导出（一般不用改）
# ============================================================
if DO_EXPORT:
    roots = [GEN_OUTPUT_DIR]
    if os.path.isdir(MISSING_OUTPUT_DIR):
        roots.append(MISSING_OUTPUT_DIR)
    if not any(os.path.isdir(r) for r in roots):
        print("[Cell6] 未找到生成目录，跳过:", roots)
    else:
        os.makedirs(EXPORTS_DIR, exist_ok=True)
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        zip_name = "%s_%s_%s.zip" % (EXPORT_PREFIX, FONT_TAG, stamp)
        zip_path = os.path.join(EXPORTS_DIR, zip_name)

        count = 0
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root_dir in roots:
                for root, _, files in os.walk(root_dir):
                    for f in sorted(files):
                        if f.lower().endswith((".png", ".jpg", ".jpeg")):
                            full = os.path.join(root, f)
                            zf.write(full, os.path.relpath(full, PROJECT_ROOT))
                            count += 1
            if EXPORT_INCLUDE_CHECKPOINT:
                ck = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
                if os.path.exists(ck):
                    zf.write(ck, "checkpoint/checkpoint-last.pth")

        size_mb = os.path.getsize(zip_path) / 1024 / 1024
        print("[Cell6] 导出完成: %s（%d 张图片, %.1f MB）" % (zip_path, count, size_mb))

        if size_mb < 1000:
            with open(zip_path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            from IPython.display import HTML, display
            display(HTML(
                '<a href="data:application/zip;base64,%s" download="%s" '
                'style="font-size:18px;background:#0d6efd;color:#fff;'
                'padding:10px 20px;text-decoration:none;border-radius:6px">'
                '&#128229; 点击下载 %s（%.1f MB）</a>'
                % (b64, zip_name, zip_name, size_mb)))
        else:
            print("[Cell6] zip 较大，请直接在左侧文件树 exports/ 目录下载: %s" % zip_path)
else:
    print("[Cell6] DO_EXPORT=False，跳过")

## 完成！

**结果一览：**

| 产物 | 位置 |
|---|---|
| 常规推理 PNG | `outputs/<字体>/generated_chars/` |
| 缺失字补集 PNG + `missing_chars.txt` | `outputs/<字体>/missing_chars/` |
| LoRA checkpoint | `outputs/<字体>/checkpoint-last.pth` |
| 导出 zip | `exports/` |

**调参速查：**

- 生成效果不理想：先调 `GENERATE_CFG`（加大）→ `GENERATE_SAMPLING_METHOD="heun"` → 加步数
- 风格化明显（行书/草书/手写）：`LORA_R=64`、`EPOCHS=300+`、`CFG=3.5~4.0`
- 补集效果不佳：训练时用完整 CHARSET（`TRAIN_CHARS_PER_FONT` ≥ 字符集大小、`MAX_CHARS_PER_FONT=None`）
- 显存相关：保持 `AUTO_TUNE=True`，自动算 batch_size，无需手动猜